# EPCLM-008 T4 engineering preflight
Select Tesla T4, enable Internet and grant access to GITHUB_TOKEN. Save & Run All creates a fresh run and publishes verified reports. Formal training and Test remain sealed.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import uuid

remote = 'https://github.com/ArcheLabs/mini-cells.git'
branch = 'research/epclm008-full-attention-system-benchmark'
with tempfile.TemporaryDirectory(prefix='epclm008-') as directory:
    repo = Path(directory) / 'repo'
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
    source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    sys.path.insert(0, str(repo))
    from scripts.research.publish_experiment_results import (
        ExperimentSpec, load_github_token, prepare_artifacts, push_results,
    )
    load_github_token('GITHUB_TOKEN')
    import torch
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name():
        raise RuntimeError('Select a Tesla T4 accelerator.')
    base = repo / 'research/projects/cellular-language-models/experiments/epclm-008-full-attention-system-benchmark'
    requirements = base.parent / 'epclm-004-unified-memory-10m/m4/requirements.txt'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)],
                   check=True, timeout=300)
    run_id = f'epclm008-t4-preflight-{source[:12]}-{uuid.uuid4().hex[:8]}'
    output = Path('/kaggle/working') / run_id
    env = {**os.environ, 'OMP_NUM_THREADS': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
    error = None
    try:
        subprocess.run([sys.executable, str(base / 'run_epclm008.py'), '--stage', 'preflight',
                        '--device', 'cuda', '--output', str(output)],
                       cwd=repo, env=env, check=True, timeout=1800)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        error = str(exc)
    output.mkdir(exist_ok=True)
    decision_path = output / 'decision.json'
    decision = json.loads(decision_path.read_text()) if decision_path.exists() else {
        'status': 'INCONCLUSIVE', 'engineering_preflight': 'FAIL',
        'error': error or 'Missing preflight decision', 'test': 'SEALED',
    }
    result_branch = f'kaggle/{run_id}'
    decision.update(format='minicells.epclm008.preflight.v1', run_id=run_id,
                    source_commit=source, gpu=torch.cuda.get_device_name(),
                    gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory,
                    branch=result_branch)
    decision_path.write_text(json.dumps(decision, indent=2))
    stage = repo / 'results/epclm008-preflight'
    stage.mkdir(parents=True)
    for report in output.glob('*.json'):
        shutil.copy2(report, stage / report.name)
    spec = ExperimentSpec('results/epclm008-preflight', f'artifacts/experiments/epclm008/{run_id}',
                          tuple(sorted(p.name for p in stage.iterdir())),
                          result_branch, decision['format'])
    try:
        destination = prepare_artifacts(repo, 'epclm008', spec, None)
        push_results(repo, destination, 'epclm008', result_branch, 'GITHUB_TOKEN', 'origin/main')
        sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
        receipt = {'status': 'REMOTE_SHA_VERIFIED', 'branch': result_branch, 'commit': sha}
    except Exception:
        (output / 'publication.json').write_text(json.dumps({'status': 'PUBLICATION_FAILED'}))
        raise
    (output / 'publication.json').write_text(json.dumps(receipt, indent=2))
    print(receipt, flush=True)
    if error or decision['engineering_preflight'] != 'PASS':
        raise RuntimeError(f'Preflight failed. Inspect {decision_path}.')
